# Steerling-8B: Logit Contribution Analysis

Steerling's interpretable architecture decomposes hidden states into three components
before projecting to vocabulary logits. For each predicted token $y$, we can measure
how much each component contributes to its logit:

$$\text{logit}(y) = \underbrace{\text{known} \cdot W_y}_{\text{known concepts}} + \underbrace{\hat{u} \cdot W_y}_{\text{discovered concepts}} + \underbrace{\varepsilon \cdot W_y}_{\text{residual}}$$

- **Known**: contribution from the top-k learned concept embeddings (human-interpretable features)
- **Discovered**: contribution from the factorized residual head (captures what known concepts miss)
- **Epsilon**: small correction term that preserves reconstruction fidelity

**Requirements:** Interpretable Steerling model, GPU with >= 18 GB VRAM

## 1. Load Model

In [ ]:
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator

model_id = "guidelabs/steerling-8b"




model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

print(generator)
print(f"Interpretable: {generator.is_interpretable}")

## 2. Decomposition Function

For each generated token, we run a full forward pass to get
`known_predicted`, `unk_hat`, and `unk_for_lm`. We then compute each component's dot product
with the LM head row $W_y$ for the predicted token $y$, giving us the raw logit contribution
and the absolute fraction each component is responsible for.

The three components are: **known** (supervised concepts), **discovered** (learned residual concepts),
and **epsilon** (reconstruction residual).

In [ ]:
@torch.no_grad()
def logit_contribution(generator, prompt, gen_length=128, steps=128, temperature=0.0):
    """
    Generate text and decompose each predicted token's logit into
    known, discovered (unk_hat), and epsilon contributions.
    """
    # Generate
    output = generator.generate(prompt, gen_length=gen_length, steps=steps, temperature=temperature)
    prompt_ids = tokenizer.encode(prompt)
    prompt_len = len(prompt_ids)
    text = generator.decode(output, prompt_len=prompt_len)

    # Forward pass with full outputs on the generated sequence
    logits, outputs = generator.model(output,  minimal_output=False)

    # Extract components for the generated region
    gen_slice = slice(prompt_len, output.shape[1])
    known = outputs.known_predicted[0, gen_slice]    # (T_gen, D)
    discovered = outputs.unk_hat[0, gen_slice] if outputs.unk_hat is not None else torch.zeros_like(known)
    unk_for_lm = outputs.unk_for_lm[0, gen_slice]   # (T_gen, D)
    epsilon = unk_for_lm - discovered                # (T_gen, D)

    # LM head weight
    lm_weight = generator.model.transformer.lm_head.weight  # (V, D)

    # Predicted token IDs in generated region
    pred_ids = output[0, gen_slice]  # (T_gen,)
    mask_id = generator.mask_token_id
    valid = pred_ids != mask_id

    # W_y for each predicted token: (T_gen, D)
    W_y = lm_weight[pred_ids].float()

    # Dot products: contribution of each component to the predicted token's logit
    known_c = (known.float() * W_y).sum(dim=-1)       # (T_gen,)
    discovered_c = (discovered.float() * W_y).sum(dim=-1)
    eps_c = (epsilon.float() * W_y).sum(dim=-1)

    # Absolute fractions
    abs_sum = known_c.abs() + discovered_c.abs() + eps_c.abs()
    abs_sum = abs_sum.clamp(min=1e-8)

    return {
        "text": text,
        "prompt": prompt,
        "token_ids": pred_ids[valid].cpu(),
        "tokens": [tokenizer.decode([t]) for t in pred_ids[valid].tolist()],
        "known_frac": (known_c.abs() / abs_sum)[valid].cpu().numpy(),
        "discovered_frac": (discovered_c.abs() / abs_sum)[valid].cpu().numpy(),
        "eps_frac": (eps_c.abs() / abs_sum)[valid].cpu().numpy(),
        "known_c": known_c[valid].cpu().numpy(),
        "discovered_c": discovered_c[valid].cpu().numpy(),
        "eps_c": eps_c[valid].cpu().numpy(),
    }

## 3. Per-Token Breakdown

The table below shows the fractional contribution of each component for every generated token.
Tokens where **Known** dominates are well-explained by the learned concept vocabulary;
tokens where **Discovered** dominates rely on features the known concepts don't capture.

In [ ]:
def show_contribution(result, max_tokens=40):
    """Print a compact summary of logit contributions."""
    print(f"Prompt:    {result['prompt']}")
    print(f"Generated: {result['text'][:200]}")
    print()

    n = min(len(result['tokens']), max_tokens)
    kf = result['known_frac'][:n]
    df = result['discovered_frac'][:n]
    ef = result['eps_frac'][:n]

    print(f"{'Token':<15} {'Known':>7} {'Disc.':>7} {'Eps':>7}")
    print("-" * 40)
    for i in range(n):
        tok = repr(result['tokens'][i])
        print(f"{tok:<15} {kf[i]:6.1%} {df[i]:6.1%} {ef[i]:6.1%}")

    print("-" * 40)
    print(f"{'Mean':<15} {kf.mean():6.1%} {df.mean():6.1%} {ef.mean():6.1%}")

In [ ]:
torch.manual_seed(42)

result = logit_contribution(generator, "The key to understanding artificial intelligence is")
show_contribution(result)

## 4. Overall Contribution

A stacked bar chart showing the mean contribution of each component across all generated tokens.
This gives a quick summary of how much the model relies on known concepts vs discovered concepts
for a given prompt.

In [ ]:
import matplotlib.pyplot as plt

PURPLE = "#675BF2"
PURPLE_LIGHT = "#ceb4fe"
GOLD = "#e2bc26"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: per-token stacked bar ---
n = min(len(result["tokens"]), 40)
tokens = [result["tokens"][i][:8] for i in range(n)]
x = np.arange(n)

axes[0].bar(x, result["known_frac"][:n], label="Known", color=PURPLE)
axes[0].bar(x, result["discovered_frac"][:n], bottom=result["known_frac"][:n], label="Discovered", color=PURPLE_LIGHT)
axes[0].bar(
    x,
    result["eps_frac"][:n],
    bottom=result["known_frac"][:n] + result["discovered_frac"][:n],
    label="Epsilon",
    color=GOLD,
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tokens, rotation=90, fontsize=7)
axes[0].set_ylabel("Fraction of |logit|")
axes[0].set_title("Per-Token Logit Contribution")
axes[0].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)
axes[0].set_ylim(0, 1.05)

# --- Right: overall mean bar ---
means = [result["known_frac"].mean(), result["discovered_frac"].mean(), result["eps_frac"].mean()]
labels = ["Known", "Discovered", "Epsilon"]
colors = [PURPLE, PURPLE_LIGHT, GOLD]

bars = axes[1].bar(labels, means, color=colors, width=0.5)
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{val:.1%}",
                 ha="center", fontsize=11)
axes[1].set_ylabel("Mean Fraction of |logit|")
axes[1].set_title("Average Contribution Across All Tokens")
axes[1].set_ylim(0, max(means) * 1.25)

fig.suptitle(f'Prompt: "{result["prompt"]}"', fontsize=11)
plt.tight_layout()
plt.show()